[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HisameOgasahara/deep-learning-diagnostics-and-improvement/blob/main/practice/08_attention_core.ipynb)

# 08. Attention core — MHA, MQA, GQA, and paper-faithful MLA

이 노트북은 일반 attention에서 시작해 KV-cache 구조를 줄이는 MQA/GQA와 DeepSeek-V2의 MLA까지 연결한다.

이번 버전의 MLA는 작은 dimension을 쓰지만 다음 계산 그래프는 유지한다.

- low-rank query compression `h -> c_Q -> q`
- joint KV compression `h -> c_KV -> k_nope, v`
- shared positional key path
- decoupled RoPE
- causal attention
- output projection
- cache absorption의 계산 동치 확인


In [ ]:
import math

import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(7)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("device:", device)


## 1. Scaled dot-product attention

Attention의 기본 계산은

`softmax(Q K^T / sqrt(d)) V`

이다.


In [ ]:
q = torch.tensor(
    [[[[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]]]],
    device=device,
)

k = q.clone()

v = torch.tensor(
    [[[[1.0, 2.0], [3.0, 4.0], [5.0, 6.0]]]],
    device=device,
)

scores = q @ k.transpose(-2, -1)
scores = scores / math.sqrt(q.size(-1))

weights = scores.softmax(dim=-1)
output = weights @ v

print("scores:")
print(scores)
print("weights:")
print(weights)
print("output:")
print(output)


## 2. MHA, MQA, and GQA differ in stored K/V heads

MHA는 query head마다 K/V를 저장하고, MQA는 하나의 K/V head를 모든 query head가 공유한다. GQA는 그 중간 형태다.

Decoding에서는 이 차이가 KV-cache 크기에 직접 연결된다.


In [ ]:
batch_size = 1
sequence_length = 6
head_dim = 8
num_query_heads = 4

q = torch.randn(
    batch_size,
    num_query_heads,
    sequence_length,
    head_dim,
    device=device,
)

k_mha = torch.randn_like(q)
v_mha = torch.randn_like(q)

k_mqa_stored = torch.randn(
    batch_size,
    1,
    sequence_length,
    head_dim,
    device=device,
)
v_mqa_stored = torch.randn_like(k_mqa_stored)

num_kv_heads = 2
k_gqa_stored = torch.randn(
    batch_size,
    num_kv_heads,
    sequence_length,
    head_dim,
    device=device,
)
v_gqa_stored = torch.randn_like(k_gqa_stored)

print("MHA stored K elements:", k_mha.numel())
print("MQA stored K elements:", k_mqa_stored.numel())
print("GQA stored K elements:", k_gqa_stored.numel())


## 3. RoPE helper for the positional subspace

MLA에서는 content key와 position-dependent RoPE key를 분리한다. 아래 helper는 마지막 dimension의 channel pair를 회전한다.


In [ ]:
def apply_rope(x, positions):
    dim = x.size(-1)

    assert dim % 2 == 0

    pair_index = torch.arange(
        0,
        dim,
        2,
        device=x.device,
        dtype=torch.float32,
    )

    inverse_frequency = 1.0 / (
        10000 ** (pair_index / dim)
    )

    angles = (
        positions.float()[:, None]
        * inverse_frequency[None, :]
    )

    cos = angles.cos()[None, None]
    sin = angles.sin()[None, None]

    even = x[..., 0::2]
    odd = x[..., 1::2]

    rotated_even = even * cos - odd * sin
    rotated_odd = even * sin + odd * cos

    return torch.stack(
        [rotated_even, rotated_odd],
        dim=-1,
    ).flatten(-2)


## 4. Tiny MLA module with both Q and KV compression

DeepSeek-V2 MLA의 핵심 low-rank paths를 작은 dimension으로 그대로 만든다.

Query path:

`h -> W_DQ -> RMSNorm -> W_UQ -> [q_nope, q_rope]`

KV path:

`h -> [c_KV, k_rope]`

`c_KV -> RMSNorm -> W_UKV -> [k_nope, v]`

RoPE는 `q_rope`와 shared `k_rope`에만 적용한다.


In [ ]:
class TinyMLA(nn.Module):
    def __init__(
        self,
        model_dim=32,
        num_heads=4,
        q_rank=12,
        kv_rank=8,
        qk_nope_dim=6,
        qk_rope_dim=2,
        value_dim=6,
    ):
        super().__init__()

        self.model_dim = model_dim
        self.num_heads = num_heads
        self.q_rank = q_rank
        self.kv_rank = kv_rank
        self.qk_nope_dim = qk_nope_dim
        self.qk_rope_dim = qk_rope_dim
        self.value_dim = value_dim

        self.q_down = nn.Linear(
            model_dim,
            q_rank,
            bias=False,
        )
        self.q_norm = nn.RMSNorm(q_rank)
        self.q_up = nn.Linear(
            q_rank,
            num_heads * (
                qk_nope_dim + qk_rope_dim
            ),
            bias=False,
        )

        self.kv_down_with_rope = nn.Linear(
            model_dim,
            kv_rank + qk_rope_dim,
            bias=False,
        )
        self.kv_norm = nn.RMSNorm(kv_rank)
        self.kv_up = nn.Linear(
            kv_rank,
            num_heads * (
                qk_nope_dim + value_dim
            ),
            bias=False,
        )

        self.out_projection = nn.Linear(
            num_heads * value_dim,
            model_dim,
            bias=False,
        )

    def project(self, hidden):
        batch_size, sequence_length, _ = hidden.shape

        q_latent = self.q_norm(
            self.q_down(hidden)
        )
        q = self.q_up(q_latent)
        q = q.view(
            batch_size,
            sequence_length,
            self.num_heads,
            self.qk_nope_dim + self.qk_rope_dim,
        ).transpose(1, 2)

        q_nope, q_rope = q.split(
            [
                self.qk_nope_dim,
                self.qk_rope_dim,
            ],
            dim=-1,
        )

        kv_and_rope = self.kv_down_with_rope(
            hidden
        )

        kv_latent, key_rope_shared = (
            kv_and_rope.split(
                [
                    self.kv_rank,
                    self.qk_rope_dim,
                ],
                dim=-1,
            )
        )

        normalized_kv = self.kv_norm(
            kv_latent
        )

        kv = self.kv_up(normalized_kv)
        kv = kv.view(
            batch_size,
            sequence_length,
            self.num_heads,
            self.qk_nope_dim + self.value_dim,
        ).transpose(1, 2)

        key_nope, value = kv.split(
            [
                self.qk_nope_dim,
                self.value_dim,
            ],
            dim=-1,
        )

        key_rope = key_rope_shared[
            :,
            None,
            :,
            :,
        ].expand(
            -1,
            self.num_heads,
            -1,
            -1,
        )

        return {
            "q_latent": q_latent,
            "q_nope": q_nope,
            "q_rope": q_rope,
            "kv_latent": kv_latent,
            "normalized_kv": normalized_kv,
            "key_nope": key_nope,
            "key_rope": key_rope,
            "value": value,
        }

    def forward(self, hidden):
        sequence_length = hidden.size(1)
        positions = torch.arange(
            sequence_length,
            device=hidden.device,
        )

        tensors = self.project(hidden)

        q_rope = apply_rope(
            tensors["q_rope"],
            positions,
        )
        key_rope = apply_rope(
            tensors["key_rope"],
            positions,
        )

        query = torch.cat(
            [
                tensors["q_nope"],
                q_rope,
            ],
            dim=-1,
        )
        key = torch.cat(
            [
                tensors["key_nope"],
                key_rope,
            ],
            dim=-1,
        )

        scores = (
            query
            @ key.transpose(-2, -1)
        )
        scores = scores / math.sqrt(
            self.qk_nope_dim
            + self.qk_rope_dim
        )

        causal_mask = torch.tril(
            torch.ones(
                sequence_length,
                sequence_length,
                dtype=torch.bool,
                device=hidden.device,
            )
        )

        scores = scores.masked_fill(
            ~causal_mask[None, None],
            torch.finfo(scores.dtype).min,
        )

        weights = scores.softmax(dim=-1)

        attended = (
            weights
            @ tensors["value"]
        )
        attended = attended.transpose(
            1,
            2,
        ).contiguous()

        attended = attended.view(
            hidden.size(0),
            sequence_length,
            self.num_heads * self.value_dim,
        )

        output = self.out_projection(
            attended
        )

        tensors["query"] = query
        tensors["key"] = key
        tensors["weights"] = weights

        return output, tensors


mla = TinyMLA().to(device)

hidden = torch.randn(
    2,
    6,
    32,
    device=device,
)

mla_output, mla_tensors = mla(hidden)

print("MLA output:", mla_output.shape)
print(
    "query latent:",
    mla_tensors["q_latent"].shape,
)
print(
    "KV latent:",
    mla_tensors["kv_latent"].shape,
)


## 5. Stored cache size

Naive MHA는 token마다 모든 head의 K와 V를 저장한다.

MLA에서는 content에 대해 `c_KV` 하나와 작은 shared RoPE key를 저장하면 된다.


In [ ]:
batch_size = hidden.size(0)
sequence_length = hidden.size(1)

naive_kv_elements = (
    batch_size
    * sequence_length
    * mla.num_heads
    * (
        mla.qk_nope_dim
        + mla.qk_rope_dim
        + mla.value_dim
    )
)

mla_cached_elements = (
    mla_tensors["kv_latent"].numel()
    + batch_size
    * sequence_length
    * mla.qk_rope_dim
)

print(
    "naive full KV-like elements:",
    naive_kv_elements,
)
print(
    "MLA cached elements:",
    mla_cached_elements,
)


## 6. Key/value absorption equivalence

MLA의 중요한 inference 관점은 decompressed `k_nope`와 `v`를 반드시 명시적으로 만들 필요가 없다는 것이다.

`k_nope = c_KV W_UK^T` 이므로

`q_nope k_nope^T = (q_nope W_UK) c_KV^T`

로 key up-projection을 query 쪽에 흡수할 수 있다.

Value path도 head별 `W_UV`와 최종 `W_O`를 미리 곱해 latent read에서 바로 model dimension으로 보낼 수 있다.

아래에서는 원래 계산과 absorbed 계산의 최대 오차를 직접 확인한다.


In [ ]:
with torch.no_grad():
    projected = mla.project(hidden)

    positions = torch.arange(
        hidden.size(1),
        device=device,
    )

    q_rope = apply_rope(
        projected["q_rope"],
        positions,
    )
    k_rope = apply_rope(
        projected["key_rope"],
        positions,
    )

    normalized_kv = projected[
        "normalized_kv"
    ]

    direct_content_scores = (
        projected["q_nope"]
        @ projected["key_nope"].transpose(-2, -1)
    )

    absorbed_content_scores = []

    for head_id in range(mla.num_heads):
        start = head_id * (
            mla.qk_nope_dim
            + mla.value_dim
        )
        stop = start + mla.qk_nope_dim

        W_UK = mla.kv_up.weight[
            start:stop,
            :
        ]

        q_absorbed = (
            projected["q_nope"][:, head_id]
            @ W_UK
        )

        score_head = (
            q_absorbed
            @ normalized_kv.transpose(-2, -1)
        )

        absorbed_content_scores.append(
            score_head
        )

    absorbed_content_scores = torch.stack(
        absorbed_content_scores,
        dim=1,
    )

    positional_scores = (
        q_rope
        @ k_rope.transpose(-2, -1)
    )

    direct_scores = (
        direct_content_scores
        + positional_scores
    )
    absorbed_scores = (
        absorbed_content_scores
        + positional_scores
    )

    scale = math.sqrt(
        mla.qk_nope_dim
        + mla.qk_rope_dim
    )

    direct_scores = direct_scores / scale
    absorbed_scores = absorbed_scores / scale

    sequence_length = hidden.size(1)

    causal_mask = torch.tril(
        torch.ones(
            sequence_length,
            sequence_length,
            dtype=torch.bool,
            device=device,
        )
    )

    direct_scores = direct_scores.masked_fill(
        ~causal_mask[None, None],
        torch.finfo(direct_scores.dtype).min,
    )
    absorbed_scores = absorbed_scores.masked_fill(
        ~causal_mask[None, None],
        torch.finfo(absorbed_scores.dtype).min,
    )

    direct_weights = direct_scores.softmax(
        dim=-1
    )
    absorbed_weights = absorbed_scores.softmax(
        dim=-1
    )

    direct_head_output = (
        direct_weights
        @ projected["value"]
    )

    direct_concat = direct_head_output.transpose(
        1,
        2,
    ).contiguous().view(
        hidden.size(0),
        sequence_length,
        mla.num_heads * mla.value_dim,
    )

    direct_output = mla.out_projection(
        direct_concat
    )

    absorbed_output = torch.zeros(
        hidden.size(0),
        sequence_length,
        mla.model_dim,
        device=device,
    )

    for head_id in range(mla.num_heads):
        kv_start = head_id * (
            mla.qk_nope_dim
            + mla.value_dim
        )
        value_start = (
            kv_start
            + mla.qk_nope_dim
        )
        value_stop = (
            value_start
            + mla.value_dim
        )

        W_UV = mla.kv_up.weight[
            value_start:value_stop,
            :
        ]

        out_start = (
            head_id
            * mla.value_dim
        )
        out_stop = (
            out_start
            + mla.value_dim
        )

        W_O_head = (
            mla.out_projection.weight[
                :,
                out_start:out_stop,
            ]
        )

        absorbed_value_out = (
            W_O_head
            @ W_UV
        )

        latent_read = (
            absorbed_weights[:, head_id]
            @ normalized_kv
        )

        absorbed_output = (
            absorbed_output
            + latent_read
            @ absorbed_value_out.T
        )

print(
    "max score error:",
    (
        direct_scores
        - absorbed_scores
    ).abs().max().item(),
)
print(
    "max output error:",
    (
        direct_output
        - absorbed_output
    ).abs().max().item(),
)


## 7. Gradient-flow sanity check

Query compression, KV compression, RoPE-separated attention, output projection까지 모두 loss에 연결되는지 확인한다.


In [ ]:
mla.zero_grad(set_to_none=True)

output, _ = mla(hidden)

loss = output.square().mean()
loss.backward()

for name in [
    "q_down.weight",
    "q_up.weight",
    "kv_down_with_rope.weight",
    "kv_up.weight",
    "out_projection.weight",
]:
    parameter = dict(
        mla.named_parameters()
    )[name]

    print(
        name,
        parameter.grad.norm().item(),
    )


## References and provenance

**MHA / MQA / GQA** — stored KV heads 수가 decoding cache에 미치는 차이를 보여준다.

**DeepSeek-V2 MLA** — low-rank query compression, low-rank joint KV compression, decoupled RoPE, shared positional key, causal attention과 output projection을 반영했다.

또한 `W_UK`를 query 쪽에, `W_UV`를 output projection 쪽에 흡수하는 계산 동치를 작은 tensor로 검증한다. 실제 serving kernel의 fused implementation은 이 노트북의 범위가 아니다.
